In [ ]:
import os
import subprocess
import glob
import requests
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from tqdm import tqdm
import json
import re
import itertools
import concurrent
import utils # Custom python utility functions

data_dir = 'batches_10k'
files = glob.glob(f'./{data_dir}/*')  # Lists all files and folders
files = [f for f in files if os.path.isfile(f)]  # Filter only files

# Number of workers used to upload files to Solr for indexing. Set max_workers=num_vCPU/8.
max_workers = 4
# model_name = 'hnsw' # Choose 'cuvs' or 'hnsw'
model_name = 'cuvs' # Choose 'cuvs' or 'hnsw'
jvm_mem = '16G'
solr_url = 'http://localhost:8983/solr/test/select'
dim = 2048

# Specify search parameters:
topK = 10
# return_limit = topK
num_queries = 8192 # Up to 8192 pre-computed query vectors

# # Load query vectors from javabin file. Each file ~489MB contains 50k records at fp64, 2048D.
# query_vector_ids, query_vectors = utils.load_javabin_data(f'{data_dir}/wiki_queries_over_1M.javabin', row_limit=num_queries)
# query_vector_strs = ['[' + ','.join(str(element) for element in v) + ']' for v in query_vectors]

# Load query vectors from parquet files (~95MB for 8192 vectors at fp64, 2048D)
query_vectors = pq.read_table('wiki_queries_over_1M.parquet', columns=['article_vector'])[:num_queries]

# Format vector as string representation for Solr
query_vector_strs = [str(v).replace(' ', '') for v in query_vectors['article_vector']]

In [ ]:
# Parameter templates:
# https://github.com/rapidsai/cuvs/blob/branch-25.08/python/cuvs_bench/cuvs_bench/config/algos/hnswlib.yaml
# https://github.com/rapidsai/cuvs/blob/branch-25.08/python/cuvs_bench/cuvs_bench/config/algos/cuvs_cagra.yaml
algo_params = {
    'hnsw': {
        'base': {
            'build': {
                'hnswMaxConnections': [2, 4],
                'hnswBeamWidth': [8, 16]
                # 'hnswMaxConnections': [12, 16, 24, 36],
                # 'hnswBeamWidth': [64, 128, 256, 512]
            }, 
            'search': {
                'ef': []
            }
        },
        'test': {
            'build': {
                'hnswMaxConnections': [32],
                'hnswBeamWidth': [512]
            }, 
            'search': {
                'ef': []
            }
        }
    },
    'cuvs': {
        'base': {
            'build': {
                'graphDegree': [32, 64, 96, 128],
                'intGraphDegree': [32, 64, 96, 128],
                'cuvsWriterThreads': [8]
            }, 
            'search': {
                'cagraITopK': [32, 64, 128, 256, 512],
                'cagraSearchWidth': [1, 2, 4, 8, 16, 32, 64]
            }
        },
        'test': {
            'build': {
                'graphDegree': [32],
                'intGraphDegree': [64],
                'cuvsWriterThreads': [8]
            }, 
            'search': {
                'cagraITopK': [32],
                'cagraSearchWidth': [1]
            }
        }
    }
}

algo_params

# Configure Solr

In [ ]:
mode = 'test' # Choose 'test' or 'base'
stage = 'build' # Choose 'build' or 'search'

# Example 
algo_params[model_name][mode][stage]

In [ ]:
def start_solr():
    start_time = time.perf_counter()
    
    # Generate solr xml config files
    utils.generate_config_xml(model_name, dim, **model_params)
    
    # Generate Solr bash scripts
    utils.generate_solr_bash_scripts(data_dir, model_name, jvm_mem)
        
    # Start Solr and reset database
    subprocess.run("chmod +x *.sh", shell=True, executable="/bin/bash")
    print()
    subprocess.run("sh ./start_solr_mod.sh", shell=True, executable="/bin/bash")
    print()

    elapsed_time = time.perf_counter() - start_time
    return(elapsed_time)

def upload_and_index_files():
    # Use python requests to submit individual javabin files to Solr in batches
    start_time = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # executor.map returns an iterator of results
        results = executor.map(utils.upload_and_index_file, files)

        # Show printout for status of files within batch
        # for file, result in zip(files, results):
        #     print(f"  Status code for {file}: {result}.")
            
    elapsed_time = time.perf_counter() - start_time
    print(f"All threads completed in {elapsed_time:.2f} s.")
    print()
    return(elapsed_time)

def param_combinations(param_dict):
    """
    Given a dictionary where each value is a list of possible values,
    return a list of dictionaries, each representing a unique combination.
    """
    keys = list(param_dict.keys())
    values = [param_dict[k] for k in keys]
    combos = itertools.product(*values)
    return [dict(zip(keys, combo)) for combo in combos]

In [ ]:
from functools import wraps

def time_it(func: any):
    """returns result and elapsed time"""
    @wraps(func)
    def inner(*args, **kwargs):
        pref = time.perf_counter()
        result = func(*args, **kwargs)
        delta = time.perf_counter() - pref
        return result, delta
    return(inner)
    
def get_system_utilization(func: any):
    @wraps(func)
    def run_function(*args, **kwargs):
        def monitor_resources(stop_event, log):
            """
            Monitor CPU and RAM usage every 1 second until stop_event is set.
            """
            while not stop_event.is_set():
                cpu = psutil.cpu_percent(interval=0.1)  # Measures over 0.1 second [mininum]
                ram = psutil.virtual_memory().used/(1024*1024*1024)
                log.append({'timestamp': time.time(), 'cpu_percent': cpu, 'ram_gb': ram})
                
        resource_log = []
        stop_event = threading.Event()
        monitor_thread = threading.Thread(target=monitor_resources, args=(stop_event, resource_log))
    
        # Start monitoring resources
        monitor_thread.start()
    
        # Run your function (blocking)
        result = func(*args, **kwargs)
    
        # Signal the monitor to stop and wait for it to finish
        stop_event.set()
        monitor_thread.join()
    
        # Process telemetry
        resource_log_df = pd.DataFrame(resource_log)
        max_ram_gb = results_df['ram_gb'].max()
        max_cpu_percent = results_df['cpu_percent'].max()
        avg_cpu_percent = results_df['cpu_percent'].mean()
        return(run_function, resource_log_df)
    return(run_function)

# get_system_utilization(time.sleep(5))

# param_combinations(algo_params['hnsw']['base']['build'])

In [ ]:
import threading
import psutil

def monitor_resources(stop_event, log):
    """
    Monitor CPU and RAM usage every 1 second until stop_event is set.
    """
    while not stop_event.is_set():
        cpu = psutil.cpu_percent(interval=1)  # Measures over 1 second
        ram = psutil.virtual_memory().used/(1024*1024*1024)
        log.append({'timestamp': time.time(), 'cpu_percent': cpu, 'ram_gb': ram})
        # print(f"CPU: {cpu:.2f}% | RAM: {ram:.2f}GB")

def run_main_function(ii, model_name, model_params, reset_solr=True):
    resource_log = []
    stop_event = threading.Event()
    monitor_thread = threading.Thread(target=monitor_resources, args=(stop_event, resource_log))

    # Start solr and reset to clean state. Don't include timing.
    if reset_solr:
        start_solr()
        print()

    # Start monitoring resources
    monitor_thread.start()
    
    # Run your function (blocking)
    print(f'**STARTING RUN {ii}:', model_params)
    
    # Run upload and indexing for a single test point
    idx_build_time = upload_and_index_files()
    
    # Signal the monitor to stop and wait for it to finish
    stop_event.set()
    monitor_thread.join()

    # Run search loops here. Don't monitor telemetry.
    

    # Process telemetry
    results_df = pd.DataFrame(resource_log)
    max_ram_gb = results_df['ram_gb'].max()
    max_cpu_percent = results_df['cpu_percent'].max()
    avg_cpu_percent = results_df['cpu_percent'].mean()

    run_metrics = {'run_num': ii, 'model_name': model_name, 'idx_build_time': idx_build_time, **model_params, 
                   'idx_build_avg_cpu_percent': avg_cpu_percent, 'idx_build_max_cpu_percent': max_cpu_percent, 
                   'idx_build_max_ram_gb': max_ram_gb
                  }

    run_summary_header = f'*** RUN {ii} SUMMARY ***'
    print('*' * len(run_summary_header))
    print(run_summary_header)
    print('*' * len(run_summary_header))
    print(model_params)
    print(f'Average CPU usage: {avg_cpu_percent:.2f}% | Peak CPU usage: {max_cpu_percent:.2f}% | Peak RAM usage: {max_ram_gb:.2f}GB')
    print('*' * 30)
    print()
    return(run_metrics)

## TODO: wrapper function to monitor telemetry at function call layer


# Run parameter sweeps
# Expand list of dict into all possible test combinations
model_params_list = param_combinations(algo_params[model_name][mode][stage])

ii = 0
run_metrics_store = []

for model_params in model_params_list[:2]:
    ii += 1
    run_metrics = run_main_function(ii, model_name, model_params)
    run_metrics_store.append(run_metrics)

In [ ]:
pd.DataFrame(run_metrics_store)

In [ ]:
%%bash
./upload_all_files_mod.sh

# Vector Search

In [ ]:
if model_name == 'hnsw':
    # Build params: hnswBeamWidth=efConstruction, hnswMaxConnections=M.
    # No efSearch parameter. Use default value.
    query_prefix = f'{{!knn f=article_vector topK={topK}}}'
elif model_name == 'cuvs':
    cagraITopK = 10
    cagraSearchWidth = 32
    query_prefix = f'{{!cuvs f=article_vector cagraITopK={cagraITopK} cagraSearchWidth={cagraSearchWidth} topK={topK}}}'
else:
    raise ValueError(f'Unknown model_name: {model_name}. Choose "hnsw" or "cuvs".')


# Run search
topK_ids = utils.run_all_queries(solr_url, query_prefix, query_vector_strs, num_queries)
topK_array = np.array(topK_ids)

# Load Ground Truth Data and Compute Recall

In [ ]:
def get_ground_truth_ids(topK, num_queries):
    """
    Load pre-computed ground truth data for specific topK, num_queries, and num_vectors.
    Determine corresponding ground_truth_ids from dataset since util.calc_truth() returns index value.
    """
    
    # Get row count in database
    url_request = solr_url+'?q=*:*&rows=0&wt=json'
    response = requests.get(url_request)
    content = json.loads(response.content.decode('utf-8'))
    num_vectors = content['response']['numFound']
    
    # Read ground truth data from disk
    filename = f'ground_truth_topK={topK}_numQueries={num_queries}_numVectors={int(num_vectors)}.csv'
    try:
        ground_truth_df = pd.read_csv(filename)
    except: 
        raise ValueError(f'Cannot load pre-computed ground truth data from {filename}.')
    
    ground_truth_ids = [np.fromstring(s.strip('[]'), sep=' ', dtype=int) for s in ground_truth_df['ids']]
    ground_truth_ids = np.array(ground_truth_ids)
    return(ground_truth_ids)

ground_truth_ids = get_ground_truth_ids(topK, num_queries)

In [ ]:
# Calculate recall using the utility function
print("Calculating recall....")
recall = utils.calc_recall(topK_array, ground_truth_ids)

print(f"\nRecall@{topK} = {recall:.4f}")

# Print some example comparisons
print("\nExample comparisons (first 3 queries):")
for i in range(5):
    print(f"\nQuery {i}:")
    print(f"Ground Truth IDs: {ground_truth_ids[i]}")
    print(f"Retrieved IDs:    {topK_ids[i]}")
    # Calculate intersection
    intersection = set(ground_truth_ids[i]) & set(topK_ids[i])
    print(f"Common IDs:       {sorted(list(intersection))}")
    print(f"Recall:          {len(intersection)/len(ground_truth_ids[i]):.4f}")